# 7장 · 큰 수의 법칙과 중심극한정리 🔔

웹 챕터 **"큰 수의 법칙과 중심극한정리"** 와 짝을 이루는 Python 실습입니다.
동전 10,000번의 수렴을 직접 그리고, **어떤 모양의 모집단이든** 표본평균의 분포가
종 모양(정규분포)이 되는 것을 시뮬레이션으로 확인합니다.

준비물: `pip install -r requirements.txt`


> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)


In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch7')

## 0. 준비: 라이브러리와 난수

시뮬레이션 결과가 설명과 똑같이 나오도록 난수 시드를 고정합니다 (`default_rng(42)`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False      # 마이너스 기호 깨짐 방지

rng = np.random.default_rng(42)                 # 시드 고정 — 실행할 때마다 같은 결과
print('준비 완료!')

## 1. 큰 수의 법칙: 동전 10,000번

동전을 던질 때마다 "지금까지의 앞면 비율"을 기록해 봅니다.
처음에는 요동치지만, 던질수록 참 확률 0.5 에 달라붙습니다.


In [ ]:
N = 10_000
flips = rng.integers(0, 2, size=N)              # 0=뒷면, 1=앞면
ratio = np.cumsum(flips) / np.arange(1, N + 1)  # 누적 앞면 비율

plt.figure(figsize=(9, 4))
plt.plot(ratio, lw=1)
plt.axhline(0.5, color='red', ls='--', label='참 확률 0.5')
plt.xscale('log')                               # 로그 눈금 — 초반 요동이 잘 보이게
plt.xlabel('던진 횟수 (로그 눈금)')
plt.ylabel('앞면 비율')
plt.title('큰 수의 법칙: 던질수록 0.5 로')
plt.legend()
plt.show()

print(f'   10번까지의 앞면 비율: {ratio[9]:.3f}')
print(f'  100번까지의 앞면 비율: {ratio[99]:.3f}')
print(f'10000번까지의 앞면 비율: {ratio[-1]:.4f}')

주사위도 똑같습니다. 눈의 평균이 기댓값 **3.5** 로 수렴해요.


In [ ]:
rolls = rng.integers(1, 7, size=N)              # 1~6 눈
avg = np.cumsum(rolls) / np.arange(1, N + 1)

plt.figure(figsize=(9, 3.5))
plt.plot(avg, lw=1, color='tab:green')
plt.axhline(3.5, color='red', ls='--', label='기댓값 3.5')
plt.xscale('log')
plt.xlabel('굴린 횟수 (로그 눈금)')
plt.ylabel('눈의 평균')
plt.title('주사위: 굴릴수록 평균이 3.5 로')
plt.legend()
plt.show()

## 2. 중심극한정리: 표본평균의 히스토그램

크기 n 표본을 **5,000번** 뽑아 표본평균 5,000개의 히스토그램을 그립니다.
n = 1 → 2 → 5 → 30 으로 키우면서 모양 변화를 보세요.

먼저 **평평한(균등) 주사위** 모집단:


In [ ]:
def sample_means(sampler, n, k=5000):
    """크기 n 표본을 k번 뽑아 표본평균 k개를 돌려준다."""
    data = sampler((k, n))
    return data.mean(axis=1)

def dice(size):
    return rng.integers(1, 7, size=size)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, n in zip(axes, [1, 2, 5, 30]):
    means = sample_means(dice, n)
    ax.hist(means, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(f'n = {n}')
    ax.set_xlim(1, 6)
fig.suptitle('주사위 표본평균의 분포: n 이 커질수록 종 모양')
plt.tight_layout()
plt.show()

이번엔 **오른쪽으로 꼬리가 긴(찌그러진)** 지수분포 모집단입니다.
원래 모양은 종과 전혀 다르지만… 표본평균은 역시 종 모양이 됩니다.


In [ ]:
def skewed(size):
    return rng.exponential(scale=2.0, size=size)   # 평균 2, 오른쪽 꼬리

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, n in zip(axes, [1, 2, 5, 30]):
    means = sample_means(skewed, n)
    ax.hist(means, bins=30, color='darkorange', edgecolor='white')
    ax.set_title(f'n = {n}')
    ax.set_xlim(0, 8)
fig.suptitle('찌그러진 분포의 표본평균: 그래도 종 모양!')
plt.tight_layout()
plt.show()

## 3. 표준오차 σ/√n: 이론과 실측 비교

표본평균의 표준편차(= 표준오차)가 정말 σ/√n 인지 확인합니다.
지수분포(scale=2)의 표준편차는 σ = 2 입니다.


In [ ]:
sigma = 2.0   # 지수분포(scale=2)의 표준편차

print(' n     실측 SE    이론 SE (σ/√n)')
print('-' * 34)
for n in [4, 16, 64, 256]:
    means = sample_means(skewed, n)
    print(f'{n:4d}   {means.std():.4f}     {sigma / np.sqrt(n):.4f}')

실측과 이론이 거의 일치하죠. **n 이 4배가 될 때마다 SE 가 절반**이 되는 것도 보입니다 (1 → 0.5 → 0.25 → 0.125).

## 정리

- **큰 수의 법칙**: 시행이 많아질수록 표본평균 → 참 평균. (동전 0.5, 주사위 3.5)
- **중심극한정리**: 모집단이 무슨 모양이든 **표본평균의 분포**는 n 이 커지면 종 모양.
- **표준오차 σ/√n**: 표본평균의 흔들림. n 4배 → 오차 절반.
- 🤖 **AI 연결**: SGD 의 미니배치 = 표본. 배치 평균 기울기가 전체를 근사하는 근거가 큰 수의 법칙입니다.

다음 노트북 `ch08_confidence_interval.ipynb`: 이 도구로 "오차범위 ±3.1%p"를 직접 만듭니다.
